# 🫀 Heart Disease Classification — Analisis & Optimasi Model

**Dataset:** Heart Statlog Cleveland Hungary Final  
**Target:** Binary — 0 (No Disease), 1 (Disease)  
**Hasil Terbaik:** Extra Trees Classifier → **89.08% Test Accuracy**

---
### Ringkasan Perbandingan Model

| Model | Test Acc | CV Acc |
|---|---|---|
| **Extra Trees** ⭐ | **89.08%** | 85.82% |
| Gradient Boosting | 87.82% | 84.56% |
| Random Forest | 87.39% | 84.98% |
| XGBoost | 86.55% | 85.09% |
| Voting Ensemble | 86.55% | 85.30% |
| SVM | 86.55% | 82.35% |
| LightGBM | 84.45% | 83.51% |


## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import (train_test_split, cross_val_score,
                                     StratifiedKFold, learning_curve)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                               GradientBoostingClassifier, VotingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import joblib

print("Libraries loaded ✅")

## 2. Load & Analisis Dataset

In [ ]:
df = pd.read_csv('data/heart_statlog_cleveland_hungary_final.csv')

print(f"Shape     : {df.shape}")
print(f"Kolom     : {df.columns.tolist()}")
print(f"\nMissing Values:\n{df.isnull().sum().to_string()}")
print(f"\nDuplikat  : {df.duplicated().sum()}")

vc = df['target'].value_counts()
print(f"\nTarget Distribution:")
print(f"  0 (No Disease) : {vc[0]} ({vc[0]/len(df)*100:.1f}%)")
print(f"  1 (Disease)    : {vc[1]} ({vc[1]/len(df)*100:.1f}%)")

In [ ]:
df.describe().round(2)

## 3. Preprocessing & Feature Engineering

In [ ]:
# Hapus duplikat
df = df.drop_duplicates(ignore_index=True)
print(f"Shape setelah drop duplikat: {df.shape}")

# Feature Engineering — fitur interaksi baru
df['pulse_pressure'] = df['resting bp s'] - df['oldpeak']        # tekanan nadi
df['heart_risk']     = df['max heart rate'] / (df['age'] + 1)    # risiko jantung
df['age_hr_ratio']   = df['age'] / (df['max heart rate'] + 1)    # rasio usia vs HR

print(f"Fitur baru ditambahkan: pulse_pressure, heart_risk, age_hr_ratio")
print(f"Shape akhir: {df.shape}")

## 4. Visualisasi EDA

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Exploratory Data Analysis — Heart Disease Dataset', fontsize=14, fontweight='bold')

sns.histplot(df['age'], kde=True, ax=axes[0,0], color='steelblue')
axes[0,0].set_title('Distribusi Usia')

sns.countplot(x='target', data=df, ax=axes[0,1], palette='Set2')
axes[0,1].set_title('Distribusi Target (0=Sehat, 1=Sakit)')

sns.boxplot(x='target', y='cholesterol', data=df, ax=axes[0,2], palette='Set2')
axes[0,2].set_title('Kolesterol vs Target')

corr = df.corr()[['target']].sort_values('target', ascending=False)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=axes[1,0], cbar=False)
axes[1,0].set_title('Korelasi Fitur vs Target')

sns.violinplot(x='target', y='max heart rate', data=df, ax=axes[1,1], palette='Set2')
axes[1,1].set_title('Max Heart Rate vs Target')

sns.countplot(x='chest pain type', hue='target', data=df, ax=axes[1,2], palette='Set2')
axes[1,2].set_title('Chest Pain Type vs Target')

plt.tight_layout()
plt.show()

## 5. Persiapan Data — Split & Scaling

In [ ]:
X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_tr = scaler.fit_transform(X_train)
X_te = scaler.transform(X_test)

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

print(f"Train : {X_train.shape}")
print(f"Test  : {X_test.shape}")
print(f"Fitur : {list(X.columns)}")

## 6. Benchmark Semua Model

In [ ]:
models = {
    'Logistic Regression' : LogisticRegression(max_iter=1000, random_state=42),
    'KNN'                 : KNeighborsClassifier(),
    'SVM'                 : SVC(C=10, gamma='scale', probability=True, random_state=42),
    'Random Forest'       : RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42),
    'Extra Trees'         : ExtraTreesClassifier(n_estimators=300, random_state=42),        # ← BEST
    'Gradient Boosting'   : GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, random_state=42),
    'XGBoost'             : XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
                                          random_state=42, eval_metric='logloss', verbosity=0),
    'LightGBM'            : LGBMClassifier(n_estimators=200, max_depth=5, random_state=42, verbose=-1),
}

print(f"{'Model':<25} {'CV Mean':>10} {'CV Std':>10} {'Test Acc':>10}")
print("-" * 58)

results = {}
for name, model in models.items():
    cv_scores = cross_val_score(model, X_tr, y_train, cv=cv, scoring='accuracy')
    model.fit(X_tr, y_train)
    test_acc = accuracy_score(y_test, model.predict(X_te))
    results[name] = {'cv_mean': cv_scores.mean(), 'cv_std': cv_scores.std(), 'test_acc': test_acc, 'model': model}
    print(f"{name:<25} {cv_scores.mean():>10.4f} {cv_scores.std():>10.4f} {test_acc:>10.4f}")

best_name = max(results, key=lambda k: results[k]['test_acc'])
print(f"\n🏆 Best Model: {best_name}  →  {results[best_name]['test_acc']*100:.2f}%")

## 7. Visualisasi Perbandingan Model

In [ ]:
names = list(results.keys())
test_accs = [results[n]['test_acc']*100 for n in names]
cv_accs   = [results[n]['cv_mean']*100  for n in names]

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(names))
w = 0.35
ax.bar(x - w/2, test_accs, w, label='Test Accuracy', color='steelblue')
ax.bar(x + w/2, cv_accs,   w, label='CV Accuracy',   color='tomato', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=25, ha='right')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Perbandingan Semua Model')
ax.set_ylim(70, 100)
ax.legend()
ax.axhline(y=max(test_accs), color='green', linestyle='--', alpha=0.6, label=f'Best: {max(test_accs):.1f}%')
for i, v in enumerate(test_accs):
    ax.text(i - w/2, v + 0.2, f'{v:.1f}%', ha='center', va='bottom', fontsize=8, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. 🏆 Best Model: Extra Trees — Evaluasi Detail

In [ ]:
best_model = results['Extra Trees']['model']
y_pred = best_model.predict(X_te)

train_acc = accuracy_score(y_train, best_model.predict(X_tr))
test_acc  = accuracy_score(y_test, y_pred)
cv_acc    = cross_val_score(best_model, X_tr, y_train, cv=cv, scoring='accuracy').mean()

print(f"{'='*40}")
print(f"  Train Accuracy : {train_acc*100:.2f}%")
print(f"  Test Accuracy  : {test_acc*100:.2f}%  ← BEST")
print(f"  CV Accuracy    : {cv_acc*100:.2f}%")
print(f"{'='*40}")
print()
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['No Disease','Disease']))

## 9. Visualisasi Hasil Best Model

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Extra Trees Classifier — Analisis Hasil', fontsize=14, fontweight='bold')

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No Disease', 'Disease'],
            yticklabels=['No Disease', 'Disease'])
axes[0].set_title('Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# Feature Importance
feat_imp = pd.Series(best_model.feature_importances_, index=X.columns).sort_values(ascending=True)
feat_imp.plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Feature Importance — Extra Trees')
axes[1].set_xlabel('Importance Score')

# Learning Curve
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
train_sizes, train_scores, test_scores = learning_curve(
    best_model, X_tr, y_train, cv=cv5, scoring='accuracy',
    train_sizes=np.linspace(0.1, 1.0, 10), n_jobs=1
)
tr_mean = train_scores.mean(axis=1)
te_mean = test_scores.mean(axis=1)
axes[2].plot(train_sizes, tr_mean, 'o-', label='Train',      color='steelblue')
axes[2].fill_between(train_sizes, tr_mean - train_scores.std(axis=1),
                     tr_mean + train_scores.std(axis=1), alpha=0.1, color='steelblue')
axes[2].plot(train_sizes, te_mean, 's-', label='Validation', color='tomato')
axes[2].fill_between(train_sizes, te_mean - test_scores.std(axis=1),
                     te_mean + test_scores.std(axis=1), alpha=0.1, color='tomato')
axes[2].set_title('Learning Curve')
axes[2].set_xlabel('Jumlah Data Training')
axes[2].set_ylabel('Accuracy')
axes[2].legend()
axes[2].set_ylim(0.6, 1.0)

plt.tight_layout()
plt.show()

## 10. Simpan Model Terbaik

In [ ]:
joblib.dump({'scaler': scaler, 'model': best_model}, 'best_model_extratrees.joblib')
print("Model disimpan: best_model_extratrees.joblib")
print(f"\n{'='*45}")
print(f"  🏆 BEST MODEL  : Extra Trees Classifier")
print(f"  🎯 Test Accuracy: {test_acc*100:.2f}%")
print(f"  📊 CV Accuracy  : {cv_acc*100:.2f}%")
print(f"{'='*45}")